In [1]:
# =====================================================================
# CELL 1: SETUP & DATA PREPARATION (N=2, 1 CHANNEL)
# =====================================================================
!pip install sentencepiece transformers safetensors accelerate scikit-learn -q

import torch
import numpy as np
import matplotlib.pyplot as plt
import json
import re
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from collections import defaultdict

embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
print("SentenceTransformer loaded.")

from google.colab import drive
drive.mount('/content/drive')

# --- CONFIGURATION ---
NUM_SEEDS_TO_USE = 2
NUM_RUNS = 5

print(f"Loading Trajectories from Google Drive (Evaluating with {NUM_SEEDS_TO_USE} seeds)...")
dataset = np.load("/content/drive/MyDrive/llada/tvs_master_variance_arrays_extra.npz")
clean_traj = dataset['clean_traj']
conflict_traj = dataset['conflict_traj']

print("Loading Verified Master Dataset for base sentences...")
with open("/content/drive/MyDrive/llada/verified_master_dataset.json", "r") as f:
    verified_eval_facts = json.load(f)

NUM_FACTS = clean_traj.shape[0]

# --- FAST BATCHED VARIANCE RECALCULATION ---
print(f"Recalculating Semantic Variance for {NUM_SEEDS_TO_USE} seeds...")
clean_raw_k, conflict_raw_k = [], []

for i in tqdm(range(NUM_FACTS), desc=f"Calculating Embeddings"):
    base_sentence = verified_eval_facts[i]["base"]

    c_traj_k = clean_traj[i, :NUM_SEEDS_TO_USE, :]
    conf_traj_k = conflict_traj[i, :NUM_SEEDS_TO_USE, :]
    K = c_traj_k.shape[0]

    c_flat = c_traj_k.T.flatten()
    conf_flat = conf_traj_k.T.flatten()
    all_texts = np.concatenate([c_flat, conf_flat])

    isolated_entities = []
    for text in all_texts:
        raw_answer = text.split(base_sentence)[-1] if base_sentence in text else text
        clean_answer = re.sub(r'[^a-zA-Z0-9\s]', '', raw_answer).strip().lower()
        isolated_entities.append(clean_answer if clean_answer else "[EMPTY_TOKEN]")

    embeddings = embedder.encode(isolated_entities, batch_size=200, show_progress_bar=False)

    c_vars, conf_vars = [], []
    for step in range(50):
        start_idx = step * K
        step_embs = embeddings[start_idx : start_idx + K]
        dist = 1 - cosine_similarity(step_embs)
        i_idx, j_idx = np.triu_indices(K, k=1)
        c_vars.append(np.clip(dist[i_idx, j_idx].mean(), 0, None))

    offset = 50 * K
    for step in range(50):
        start_idx = offset + (step * K)
        step_embs = embeddings[start_idx : start_idx + K]
        dist = 1 - cosine_similarity(step_embs)
        i_idx, j_idx = np.triu_indices(K, k=1)
        conf_vars.append(np.clip(dist[i_idx, j_idx].mean(), 0, None))

    clean_raw_k.append(c_vars)
    conflict_raw_k.append(conf_vars)

clean_raw = np.array(clean_raw_k)
conflict_raw = np.array(conflict_raw_k)

# Delta TVS (Velocity) generation removed for this ablation.
X_list, y_list = [], []
for i in range(NUM_FACTS):
    # Expand dims to simulate a single feature channel (Raw Variance Only)
    X_list.append(np.expand_dims(clean_raw[i], axis=0))
    y_list.append(0)
    X_list.append(np.expand_dims(conflict_raw[i], axis=0))
    y_list.append(1)

# Format for TraceDet LSTM: [Batch, Time, Channels] -> [Batch, 50, 1]
X = torch.tensor(np.array(X_list), dtype=torch.float32).permute(0, 2, 1)
y = torch.tensor(y_list, dtype=torch.float32).unsqueeze(1)
indices = torch.arange(len(y))

print(f"Data Prep Complete! X shape: {X.shape} (1 Channel)")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer loaded.
Mounted at /content/drive
Loading Trajectories from Google Drive (Evaluating with 2 seeds)...
Loading Verified Master Dataset for base sentences...
Recalculating Semantic Variance for 2 seeds...


Calculating Embeddings: 100%|██████████| 2947/2947 [04:58<00:00,  9.88it/s]

Data Prep Complete! X shape: torch.Size([5894, 50, 1]) (1 Channel)


In [2]:
# =====================================================================
# CELL 2: MODEL ARCHITECTURES (TRACEDET vs. LR)
# =====================================================================

# --- 1. TRACEDET BASELINE (Variational Information Bottleneck) ---
class TraceDetApproximation(nn.Module):
    # input_dim is 1 by default, matching our 1-channel data shape
    def __init__(self, input_dim=1, hidden_size=32, latent_dim=16):
        super(TraceDetApproximation, self).__init__()
        self.encoder = nn.LSTM(input_size=input_dim, hidden_size=hidden_size, batch_first=True, bidirectional=True)
        self.fc_mu = nn.Linear(hidden_size * 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_size * 2, latent_dim)
        self.classifier = nn.Sequential(nn.Linear(latent_dim, 16), nn.ReLU(), nn.Linear(16, 1))

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        else:
            return mu

    def forward(self, x):
        _, (h_n, _) = self.encoder(x)
        h_n = torch.cat((h_n[-2,:,:], h_n[-1,:,:]), dim=1)
        mu = self.fc_mu(h_n)
        logvar = self.fc_logvar(h_n)
        z = self.reparameterize(mu, logvar)
        logits = self.classifier(z)
        return logits, mu, logvar

def tracedet_loss_function(logits, labels, mu, logvar, beta=1e-3):
    bce_loss = F.binary_cross_entropy_with_logits(logits.view(-1), labels.float().view(-1))
    kl_divergence = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
    kl_loss = torch.mean(kl_divergence)
    return bce_loss + (beta * kl_loss)

# Note: Logistic Regression is handled via scikit-learn in the next cell.

In [3]:
# =====================================================================
# CELL 3: HEAD-TO-HEAD EVALUATION LOOP
# =====================================================================
BATCH_SIZE = 64
EPOCHS = 75
LEARNING_RATE = 0.002
WEIGHT_DECAY = 1e-4

results_summary = {
    "TraceDet (Baseline)": {"acc": [], "auroc": [], "ds_correct": defaultdict(int), "ds_total": defaultdict(int)},
    "Logistic Regression": {"acc": [], "auroc": [], "ds_correct": defaultdict(int), "ds_total": defaultdict(int)}
}

print(f"\n{'='*70}")
print(f"RUNNING {NUM_RUNS} PARALLEL TRIALS: TRACEDET vs. LR (1-CH, N=2)")
print(f"{'='*70}")

for run in range(NUM_RUNS):
    print(f"\n--- TRIAL {run+1}/{NUM_RUNS} ---")

    # Randomly split the data differently each run
    X_train, X_temp, y_train, y_temp, idx_train, idx_temp = train_test_split(X, y, indices, test_size=0.30, random_state=42+run, stratify=y)
    X_val, X_test, y_val, y_test, idx_val, idx_test = train_test_split(X_temp, y_temp, idx_temp, test_size=0.50, random_state=42+run, stratify=y_temp)

    y_test_np = y_test.numpy().squeeze()
    idx_test_np = idx_test.numpy()

    # -------------------------------------------------------------
    # 1. TRAIN TRACEDET (PyTorch LSTM-VIB)
    # -------------------------------------------------------------
    # X is already [Batch, 50, 1] so we can pass it directly
    td_train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
    td_val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)

    model_td = TraceDetApproximation()
    optimizer_td = optim.AdamW(model_td.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    best_val_loss_td = float('inf')
    best_state_td = None

    for epoch in range(EPOCHS):
        model_td.train()
        for bX, by in td_train_loader:
            optimizer_td.zero_grad()
            logits, mu, logvar = model_td(bX)
            loss = tracedet_loss_function(logits, by, mu, logvar)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_td.parameters(), max_norm=1.0)
            optimizer_td.step()

        model_td.eval()
        val_loss = 0.0
        with torch.no_grad():
            for bX, by in td_val_loader:
                logits, mu, logvar = model_td(bX)
                val_loss += tracedet_loss_function(logits, by, mu, logvar).item() * bX.size(0)
        avg_val_loss = val_loss / len(X_val)

        if avg_val_loss < best_val_loss_td:
            best_val_loss_td = avg_val_loss
            best_state_td = model_td.state_dict().copy()

    # Evaluate TraceDet on Test Set
    model_td.load_state_dict(best_state_td)
    model_td.eval()
    with torch.no_grad():
        test_logits, _, _ = model_td(X_test)
        test_probs_td = torch.sigmoid(test_logits).squeeze().numpy()
        test_preds_td = (test_probs_td >= 0.5).astype(float)

    acc_td = accuracy_score(y_test_np, test_preds_td)
    auroc_td = roc_auc_score(y_test_np, test_probs_td)
    results_summary["TraceDet (Baseline)"]["acc"].append(acc_td * 100)
    results_summary["TraceDet (Baseline)"]["auroc"].append(auroc_td)
    print(f"TraceDet | Test Acc: {acc_td * 100:.2f}% | Test AUROC: {auroc_td:.4f}")

    # Log per-dataset predictions for TraceDet
    for k in range(len(y_test_np)):
        ds_name = verified_eval_facts[idx_test_np[k] // 2].get("dataset", "unknown").upper()
        results_summary["TraceDet (Baseline)"]["ds_total"][ds_name] += 1
        if y_test_np[k] == test_preds_td[k]:
            results_summary["TraceDet (Baseline)"]["ds_correct"][ds_name] += 1

    # -------------------------------------------------------------
    # 2. TRAIN LOGISTIC REGRESSION (scikit-learn)
    # -------------------------------------------------------------
    # Scikit-learn requires 1D feature arrays per sample.
    # Flatten [Batch, 50, 1] -> [Batch, 50]
    X_train_lr = X_train.numpy().reshape(X_train.shape[0], -1)
    X_test_lr = X_test.numpy().reshape(X_test.shape[0], -1)
    y_train_lr = y_train.numpy().flatten()

    lr_model = LogisticRegression(max_iter=2000, random_state=42)
    lr_model.fit(X_train_lr, y_train_lr)

    test_probs_lr = lr_model.predict_proba(X_test_lr)[:, 1]
    test_preds_lr = lr_model.predict(X_test_lr)

    acc_lr = accuracy_score(y_test_np, test_preds_lr)
    auroc_lr = roc_auc_score(y_test_np, test_probs_lr)
    results_summary["Logistic Regression"]["acc"].append(acc_lr * 100)
    results_summary["Logistic Regression"]["auroc"].append(auroc_lr)
    print(f"LogReg   | Test Acc: {acc_lr * 100:.2f}% | Test AUROC: {auroc_lr:.4f}")

    # Log per-dataset predictions for Logistic Regression
    for k in range(len(y_test_np)):
        ds_name = verified_eval_facts[idx_test_np[k] // 2].get("dataset", "unknown").upper()
        results_summary["Logistic Regression"]["ds_total"][ds_name] += 1
        if y_test_np[k] == test_preds_lr[k]:
            results_summary["Logistic Regression"]["ds_correct"][ds_name] += 1

# =====================================================================
# PRINT OVERALL RESULTS
# =====================================================================
print("\n" + "="*50)
print("FINAL RESULTS: TRACEDET vs. LOGISTIC REGRESSION")
print("="*50)
for model_name, metrics in results_summary.items():
    print(f"{model_name}:")
    print(f"  Overall Accuracy : {np.mean(metrics['acc']):.2f}% ± {np.std(metrics['acc']):.2f}%")
    print(f"  Overall AUROC    : {np.mean(metrics['auroc']):.4f} ± {np.std(metrics['auroc']):.4f}\n")


RUNNING 5 PARALLEL TRIALS: TRACEDET vs. LR (1-CH, N=2)

--- TRIAL 1/5 ---
TraceDet | Test Acc: 71.64% | Test AUROC: 0.7887
LogReg   | Test Acc: 69.94% | Test AUROC: 0.7590

--- TRIAL 2/5 ---
TraceDet | Test Acc: 71.75% | Test AUROC: 0.7994
LogReg   | Test Acc: 71.07% | Test AUROC: 0.7843

--- TRIAL 3/5 ---
TraceDet | Test Acc: 70.06% | Test AUROC: 0.7772
LogReg   | Test Acc: 70.40% | Test AUROC: 0.7711

--- TRIAL 4/5 ---
TraceDet | Test Acc: 71.86% | Test AUROC: 0.7800
LogReg   | Test Acc: 71.75% | Test AUROC: 0.7760

--- TRIAL 5/5 ---
TraceDet | Test Acc: 71.41% | Test AUROC: 0.7872
LogReg   | Test Acc: 70.62% | Test AUROC: 0.7723

FINAL RESULTS: TRACEDET vs. LOGISTIC REGRESSION
TraceDet (Baseline):
  Overall Accuracy : 71.34% ± 0.66%
  Overall AUROC    : 0.7865 ± 0.0078

Logistic Regression:
  Overall Accuracy : 70.76% ± 0.62%
  Overall AUROC    : 0.7725 ± 0.0082

